# Win Model Evaluation

Overall metrics (single split + CV), then trajectory analysis showing
how the model's ability to identify the winner evolves episode-by-episode.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats
from src.models.win import (
    train_eval_pipeline, cross_validate,
    metrics_by_episode_number, winner_rank_detail,
    predict_season,
)
from src.features.build import build_modeling_table
from src.load import load_data


In [ ]:
data = load_data()
df = build_modeling_table(data)

## Overall metrics

Single train/test split (seasons 1-40 → 41-49) and expanding-window CV.
These give one number averaging over all episodes equally.

In [ ]:
print("=== Single split ===\n")
results = train_eval_pipeline(df)

print("\n\n=== Cross-validation ===\n")
cv_results = cross_validate(df)

## Accuracy trajectory (cross-validated)

In [ ]:
# Pool predictions from all CV folds (non-overlapping test seasons → ~49 seasons total)
all_cv_preds = pd.concat([fold["predictions"] for fold in cv_results["folds"]])

# Get accuracy metrics at each episode number, averaged across seasons
by_ep_cv = metrics_by_episode_number(all_cv_preds)
by_ep_cv = by_ep_cv[by_ep_cv["n_seasons"] >= 10]  # drop noisy tail episodes

MODEL_COLOR = "#0d9488"
BASELINE_COLOR = "#9ca3af"
marker_sizes = by_ep_cv["n_seasons"] * 4  # bigger dot = more seasons behind it
z95 = 1.96

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Win model accuracy over season (cross-validated)", fontsize=13, y=1.02)

# Left plot: where does the model rank the winner? (lower = better)
ax = axes[0]
ax.plot(by_ep_cv["episode"], by_ep_cv["mean_winner_rank"], "-",
        color=MODEL_COLOR, alpha=0.6, zorder=2)
ax.scatter(by_ep_cv["episode"], by_ep_cv["mean_winner_rank"], s=marker_sizes,
           color=MODEL_COLOR, label="Model (size = n seasons)", zorder=3)
ax.fill_between(
    by_ep_cv["episode"],
    by_ep_cv["mean_winner_rank"] - z95 * by_ep_cv["se_winner_rank"],
    by_ep_cv["mean_winner_rank"] + z95 * by_ep_cv["se_winner_rank"],
    color=MODEL_COLOR, alpha=0.12, zorder=1,
)
ax.plot(by_ep_cv["episode"], by_ep_cv["mean_baseline_rank"], "s--",
        color=BASELINE_COLOR, alpha=0.6, label="Random baseline")
ax.set_xlabel("Episode")
ax.set_ylabel("Mean winner rank (lower = better)")
ax.set_title("Winner rank")
ax.legend()
ax.invert_yaxis()

# Right plot: how often is the winner the model's #1 pick?
ax = axes[1]
ax.plot(by_ep_cv["episode"], by_ep_cv["top1_accuracy"], "-",
        color=MODEL_COLOR, alpha=0.6, zorder=2)
ax.scatter(by_ep_cv["episode"], by_ep_cv["top1_accuracy"], s=marker_sizes,
           color=MODEL_COLOR, label="Model (size = n seasons)", zorder=3)
ax.fill_between(
    by_ep_cv["episode"],
    (by_ep_cv["top1_accuracy"] - z95 * by_ep_cv["se_top1"]).clip(0),
    (by_ep_cv["top1_accuracy"] + z95 * by_ep_cv["se_top1"]).clip(0, 1),
    color=MODEL_COLOR, alpha=0.12, zorder=1,
)
ax.plot(by_ep_cv["episode"], by_ep_cv["baseline_top1"], "s--",
        color=BASELINE_COLOR, alpha=0.6, label="Random baseline")
ax.set_xlabel("Episode")
ax.set_ylabel("Top-1 accuracy")
ax.set_title("Top-1 accuracy")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()

fig.tight_layout()
plt.show()

print(by_ep_cv.to_string(index=False))

## Significance test

In [ ]:
# Reuse the per-(season, episode) detail already computed in win.py
detail = winner_rank_detail(all_cv_preds)

# Is the model's rank improvement over random significantly > 0?
t, p = stats.ttest_1samp(detail["rank_improvement"], 0)
print(f"Paired t-test (model rank vs baseline rank, all season-episodes):")
print(f"  Mean improvement: {detail['rank_improvement'].mean():.2f} ranks better than random")
print(f"  t = {t:.2f}, p = {p:.4f}, n = {len(detail)}")

## Winner prediction by season

For each season in the CV predictions, track how the model ranked the eventual
winner across the full season: how often they were the #1 pick, how often they
were in the top 3, their best rank, etc. This uses the app-style framing
("Ranked #1 for X of Y episodes") rather than aggregate accuracy numbers.

In [ ]:
def player_season_summary(preds: pd.DataFrame) -> pd.DataFrame:
    """For every (season, player), compute ranking stats across all their episodes.

    Returns one row per player-season with:
      - n_episodes: how many episodes they appeared in
      - eps_ranked_1, eps_top_3: how many episodes they were ranked #1 / top 3
      - frac_ranked_1, frac_top_3: same as fractions
      - best_rank: their best (lowest) rank in any episode
      - final_rank: their rank in the last episode they appeared in
      - won_season: whether they won
    """
    rng = np.random.default_rng(42)

    rows = []
    for (season, episode), grp in preds.groupby(["season", "episode"]):
        noisy = grp["prob_win"] + rng.uniform(0, 1e-10, len(grp))
        ranks = noisy.rank(ascending=False).astype(int)
        for idx, row in grp.iterrows():
            rows.append({
                "season": season,
                "episode": episode,
                "castaway_id": row["castaway_id"],
                "castaway": row["castaway"],
                "won_season": row["won_season"],
                "rank": ranks[idx],
                "n_players": len(grp),
            })

    episode_ranks = pd.DataFrame(rows)

    summary = episode_ranks.groupby(["season", "castaway_id", "castaway", "won_season"]).agg(
        n_episodes=("episode", "count"),
        eps_ranked_1=("rank", lambda x: (x == 1).sum()),
        eps_top_3=("rank", lambda x: (x <= 3).sum()),
        best_rank=("rank", "min"),
        final_rank=("rank", "last"),
        final_n_players=("n_players", "last"),
    ).reset_index()

    summary["frac_ranked_1"] = summary["eps_ranked_1"] / summary["n_episodes"]
    summary["frac_top_3"] = summary["eps_top_3"] / summary["n_episodes"]

    return summary


summary = player_season_summary(all_cv_preds)
winners = summary[summary["won_season"] == 1].sort_values("season")

print(f"Seasons with CV predictions: {winners['season'].nunique()}\n")
print("=== How the model tracked the eventual winner ===\n")
for _, w in winners.iterrows():
    print(
        f"  S{int(w['season']):>2d}  {w['castaway']:20s}  "
        f"Best rank: #{int(w['best_rank'])}  |  "
        f"Ranked #1 in {int(w['eps_ranked_1']):>2d}/{int(w['n_episodes'])} eps  |  "
        f"Top 3 in {int(w['eps_top_3']):>2d}/{int(w['n_episodes'])} eps  |  "
        f"Final rank: #{int(w['final_rank'])} of {int(w['final_n_players'])}"
    )

print(f"\n--- Aggregate across {len(winners)} seasons ---")
print(f"  Winner was ever ranked #1:           {(winners['eps_ranked_1'] > 0).mean():.0%} of seasons")
print(f"  Winner ranked #1 in final episode:   {(winners['final_rank'] == 1).mean():.0%} of seasons")
print(f"  Winner in top 3 in final episode:    {(winners['final_rank'] <= 3).mean():.0%} of seasons")
print(f"  Mean fraction of episodes at #1:     {winners['frac_ranked_1'].mean():.1%}")
print(f"  Mean fraction of episodes in top 3:  {winners['frac_top_3'].mean():.1%}")
print(f"  Mean best rank achieved:             {winners['best_rank'].mean():.1f}")

In [ ]:
# Compare winners vs other finalists (players present in the final episode).
# The winner naturally survives every episode, so they have more chances to be
# ranked highly. Restricting to finalists controls for that.

final_eps = all_cv_preds.groupby("season")["episode"].max().reset_index()
final_eps.columns = ["season", "max_episode"]

finalists = (
    all_cv_preds
    .merge(final_eps, on="season")
    .query("episode == max_episode")[["season", "castaway_id"]]
)

finalist_summary = summary.merge(finalists, on=["season", "castaway_id"], how="inner")
finalist_summary["is_winner"] = finalist_summary["won_season"].astype(int)

print(f"Finalists across {finalist_summary['season'].nunique()} seasons: "
      f"{len(finalist_summary)} players ({finalist_summary['is_winner'].sum()} winners)\n")

compare = finalist_summary.groupby("is_winner").agg(
    n=("season", "count"),
    mean_frac_ranked_1=("frac_ranked_1", "mean"),
    mean_frac_top_3=("frac_top_3", "mean"),
    mean_best_rank=("best_rank", "mean"),
    mean_eps_ranked_1=("eps_ranked_1", "mean"),
    mean_eps_top_3=("eps_top_3", "mean"),
    pct_ever_ranked_1=("eps_ranked_1", lambda x: (x > 0).mean()),
    mean_final_rank=("final_rank", "mean"),
).rename(index={0: "Other finalists", 1: "Winner"})

print(compare.T.to_string(float_format=lambda x: f"{x:.2f}"))

print("\n\n=== Which metric best separates winners from other finalists? ===\n")

for metric, label in [
    ("frac_ranked_1", "Frac. of eps ranked #1"),
    ("frac_top_3", "Frac. of eps in top 3"),
    ("best_rank", "Best rank (lower=better)"),
    ("final_rank", "Final episode rank"),
]:
    w = finalist_summary[finalist_summary["is_winner"] == 1][metric]
    nw = finalist_summary[finalist_summary["is_winner"] == 0][metric]
    t, p = stats.mannwhitneyu(w, nw, alternative="two-sided")
    direction = "higher" if w.mean() > nw.mean() else "lower"
    if metric in ("best_rank", "final_rank"):
        direction = "better" if w.mean() < nw.mean() else "worse"
    print(f"  {label:35s}  winner={w.mean():.2f}  other={nw.mean():.2f}  "
          f"({direction})  p={p:.4f}")

In [ ]:
# For each season: which finalist prediction method picks the winner?
# Three methods:
#   1. Highest frac of episodes ranked #1 (uses fractions, not raw counts,
#      so returning players like Chris in S38 aren't penalized)
#   2. Highest frac of episodes in top 3
#   3. Ranked #1 in the final episode

results_rows = []
for season, grp in finalist_summary.groupby("season"):
    winner = grp[grp["is_winner"] == 1].iloc[0]
    best_frac_1 = grp.loc[grp["frac_ranked_1"].idxmax()]
    best_frac_3 = grp.loc[grp["frac_top_3"].idxmax()]
    best_finale = grp.loc[grp["final_rank"].idxmin()]

    results_rows.append({
        "season": int(season),
        "winner": winner["castaway"],
        "best_frac_1": best_frac_1["castaway"],
        "best_frac_1_correct": best_frac_1["castaway"] == winner["castaway"],
        "best_frac_3": best_frac_3["castaway"],
        "best_frac_3_correct": best_frac_3["castaway"] == winner["castaway"],
        "finale_pick": best_finale["castaway"],
        "finale_correct": best_finale["castaway"] == winner["castaway"],
        "winner_final_rank": int(winner["final_rank"]),
        "winner_final_n": int(winner["final_n_players"]),
    })

season_results = pd.DataFrame(results_rows)

print("=== Season-level: which method picks the winner among finalists? ===\n")
print(f"  {'':4s}  {'Winner':20s}  {'Highest frac #1':22s}      {'Highest frac top-3':22s}      {'#1 at finale':22s}")
print(f"  {'':4s}  {'':20s}  {'':22s}      {'':22s}      {'(winner rank)'}")
for _, r in season_results.iterrows():
    f1 = "✓" if r["best_frac_1_correct"] else " "
    f3 = "✓" if r["best_frac_3_correct"] else " "
    ff = "✓" if r["finale_correct"] else " "
    print(
        f"  S{r['season']:>2d}  {r['winner']:20s}  "
        f"{r['best_frac_1']:20s} [{f1}]  "
        f"{r['best_frac_3']:20s} [{f3}]  "
        f"{r['finale_pick']:20s} [{ff}]  "
        f"(#{r['winner_final_rank']} of {r['winner_final_n']})"
    )

n = len(season_results)
n_finalists = finalist_summary.groupby("season").size()
baseline = (1 / n_finalists).mean()

print(f"\n--- Prediction rates across {n} seasons (baseline: {baseline:.0%}) ---")
print(f"  Highest frac of eps at #1:  "
      f"{season_results['best_frac_1_correct'].sum()}/{n} "
      f"({season_results['best_frac_1_correct'].mean():.0%})")
print(f"  Highest frac of eps top 3:  "
      f"{season_results['best_frac_3_correct'].sum()}/{n} "
      f"({season_results['best_frac_3_correct'].mean():.0%})")
print(f"  Ranked #1 at finale:        "
      f"{season_results['finale_correct'].sum()}/{n} "
      f"({season_results['finale_correct'].mean():.0%})")

## Season predictions

Train both models on all prior seasons, then predict a single target season.
Change `TARGET_SEASON` below to view a different season.

In [ ]:
TARGET_SEASON = 18

preds = predict_season(df, TARGET_SEASON)

In [ ]:
def plot_season_predictions(preds, prob_col, title, ylabel, highlight_col=None):
    """Interactive line chart of per-player probabilities over episodes."""
    plot_df = preds.copy()
    plot_df["prob_pct"] = (plot_df[prob_col] * 100).round(1)

    # Mark eliminated players in the hover text
    if highlight_col and highlight_col in plot_df.columns:
        plot_df["status"] = plot_df[highlight_col].map({1: "ELIMINATED", 0: ""})
    else:
        plot_df["status"] = ""

    fig = px.line(
        plot_df.sort_values(["castaway", "episode"]),
        x="episode", y=prob_col, color="castaway",
        markers=True,
        title=title,
        labels={prob_col: ylabel, "episode": "Episode", "castaway": "Player"},
        hover_data={"prob_pct": ":.1f", "status": True, prob_col: False},
    )

    # Add X markers for eliminated players
    if highlight_col and highlight_col in plot_df.columns:
        elim = plot_df[plot_df[highlight_col] == 1]
        if not elim.empty:
            fig.add_scatter(
                x=elim["episode"], y=elim[prob_col],
                mode="markers", marker=dict(symbol="x", size=14, color="black", line_width=2),
                name="Eliminated", showlegend=True,
                hoverinfo="skip",
            )

    fig.update_yaxes(tickformat=".0%")
    fig.update_xaxes(dtick=1)
    fig.update_layout(height=500, legend_title_text="Player")
    fig.show()


plot_season_predictions(
    preds, "prob_eliminated",
    title=f"Season {TARGET_SEASON} — Elimination risk by episode",
    ylabel="P(eliminated)",
    highlight_col="eliminated_this_episode",
)

plot_season_predictions(
    preds, "prob_win",
    title=f"Season {TARGET_SEASON} — Win probability by episode",
    ylabel="P(win)",
    highlight_col="eliminated_this_episode",
)

In [ ]:
def cumulative_rank_trajectories(preds: pd.DataFrame) -> pd.DataFrame:
    """For each player at each episode, compute running ranking stats.

    At episode E, for each remaining player: what fraction of episodes 1..E
    were they ranked #1? Top 3? What's their running average rank?
    """
    rng = np.random.default_rng(42)

    rows = []
    for (season, episode), grp in preds.groupby(["season", "episode"]):
        noisy = grp["prob_win"] + rng.uniform(0, 1e-10, len(grp))
        ranks = noisy.rank(ascending=False).astype(int)
        for idx, row in grp.iterrows():
            rows.append({
                "season": season, "episode": episode,
                "castaway_id": row["castaway_id"], "castaway": row["castaway"],
                "won_season": row["won_season"], "prob_win": row["prob_win"],
                "rank": ranks[idx], "n_players": len(grp),
                "eliminated": row.get("eliminated_this_episode", 0),
            })

    ep_ranks = pd.DataFrame(rows).sort_values(["castaway_id", "episode"])

    ep_ranks["is_rank1"] = (ep_ranks["rank"] == 1).astype(int)
    ep_ranks["is_top3"] = (ep_ranks["rank"] <= 3).astype(int)
    ep_ranks["cum_rank1"] = ep_ranks.groupby("castaway_id")["is_rank1"].cumsum()
    ep_ranks["cum_top3"] = ep_ranks.groupby("castaway_id")["is_top3"].cumsum()
    ep_ranks["ep_number"] = ep_ranks.groupby("castaway_id").cumcount() + 1
    ep_ranks["frac_rank1_so_far"] = ep_ranks["cum_rank1"] / ep_ranks["ep_number"]
    ep_ranks["frac_top3_so_far"] = ep_ranks["cum_top3"] / ep_ranks["ep_number"]
    ep_ranks["cum_avg_rank"] = (
        ep_ranks.groupby("castaway_id")["rank"]
        .expanding().mean().reset_index(level=0, drop=True)
    )

    return ep_ranks


season_ranks = cumulative_rank_trajectories(preds)
winner_name = preds.loc[preds["won_season"] == 1, "castaway"].iloc[0]

# Identify top contenders: anyone who was ever ranked #1, plus the winner
ever_rank1 = set(season_ranks.loc[season_ranks["rank"] == 1, "castaway"].unique())
highlight_players = ever_rank1 | {winner_name}
palette = plt.cm.tab10.colors
highlight_colors = {p: palette[i % len(palette)] for i, p in enumerate(sorted(highlight_players - {winner_name}))}
highlight_colors[winner_name] = "#d4a017"

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f"Season {TARGET_SEASON} — Cumulative ranking stats by episode", fontsize=13, y=1.02)

for ax, col, title, ylabel in [
    (axes[0], "frac_rank1_so_far", "Fraction of episodes ranked #1", "Frac. ranked #1 (cumulative)"),
    (axes[1], "frac_top3_so_far", "Fraction of episodes in top 3", "Frac. in top 3 (cumulative)"),
]:
    for castaway, grp in season_ranks.groupby("castaway"):
        is_highlight = castaway in highlight_players
        is_winner = castaway == winner_name
        ax.plot(
            grp["episode"], grp[col],
            color=highlight_colors.get(castaway, "#d4d4d4"),
            alpha=1.0 if is_highlight else 0.15,
            linewidth=2.5 if is_winner else (1.5 if is_highlight else 0.8),
            zorder=10 if is_winner else (5 if is_highlight else 1),
        )
        # Label highlighted players at their last episode
        if is_highlight:
            last = grp.iloc[-1]
            label = f"{castaway} ★" if is_winner else castaway
            ax.annotate(
                label, xy=(last["episode"], last[col]),
                fontsize=8, fontweight="bold" if is_winner else "normal",
                color=highlight_colors[castaway],
                xytext=(5, 0), textcoords="offset points", va="center",
            )

    ax.set_xlabel("Episode")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    ax.set_xlim(season_ranks["episode"].min() - 0.2, season_ranks["episode"].max() + 2.5)

fig.tight_layout()
plt.show()

# Leaderboard: each player's stats as of their last episode
last_ep_per_player = season_ranks.groupby("castaway_id").last().reset_index()
last_ep_per_player = last_ep_per_player.sort_values("frac_rank1_so_far", ascending=False)

print(f"\nAll-season leaderboard (each player's stats through their last episode):\n")
print(f"  {'Player':20s}  {'Last ep':>7s}  {'Frac #1':>8s}  {'Frac top3':>9s}  {'Avg rank':>8s}")
for _, r in last_ep_per_player.iterrows():
    flag = " <-- WINNER" if r["castaway"] == winner_name else ""
    print(
        f"  {r['castaway']:20s}  {int(r['episode']):>5d}    "
        f"{r['frac_rank1_so_far']:>6.0%}    "
        f"{r['frac_top3_so_far']:>6.0%}    "
        f"{r['cum_avg_rank']:>6.1f}  {flag}"
    )